# Testing RAG Engine Step-by-Step

This notebook allows for step-by-step execution and understanding of the `RagEngine` defined in `src/rag_engine.py`.

In [1]:
import sys
import os
from dotenv import load_dotenv

# Add ../src to sys.path so we can import modules from src
sys.path.append(os.path.abspath('../src'))
print("Added ../src to sys.path")

load_dotenv()
print("Environment variables loaded")

Added ../src to sys.path
Environment variables loaded


## 1. Initialization
First, we import the configuration and the `RagEngine` class, then create an instance. This step initializes the paths and the language models/embeddings.

In [2]:
from config import CONFIG
from rag_engine import RagEngine

# Initialize the engine
rag = RagEngine()
print("RagEngine initialized.")
print(f"Document language: {rag.doc_language}")
print(f"Embeddings ready: {rag.embeddings is not None}")

RagEngine initialized.
Document language: Spanish
Embeddings ready: True


## 2. Setting up the Vector Indices
The RAG engine uses two indices: one for the client documents and one for regulations. `build_index` and `ingest_regulations` will load them from FAISS indices on disk or build them from PDFs if they don't exist.

In [3]:
# Build or load the client documents index
print("Building/Loading Client Index...")
rag.build_index()

# Build or load the regulations index
print("\nBuilding/Loading Regulations Index...")
rag.ingest_regulations()

print(f"\nClient Vector Store: {rag.vector_store}")
print(f"Regulations Vector Store: {rag.vector_store_regs}")

Building/Loading Client Index...
Loading existing index from c:\Users\semeier\Desktop\AI Project\Agentic_AI_for_validation\faiss_index_client...
Index c:\Users\semeier\Desktop\AI Project\Agentic_AI_for_validation\faiss_index_client loaded successfully.

Building/Loading Regulations Index...
Loading documents from c:\Users\semeier\Desktop\AI Project\Agentic_AI_for_validation\regulations...
Loading Guidelines on Accounting for ECL (EBA-GL-2017-06)_EN.pdf...
Creating vector store for c:\Users\semeier\Desktop\AI Project\Agentic_AI_for_validation\faiss_index_regs with 163 chunks...
Processing batch 1/17 (10 chunks)...
Processing batch 2/17 (10 chunks)...
Processing batch 3/17 (10 chunks)...
Processing batch 4/17 (10 chunks)...
Processing batch 5/17 (10 chunks)...
Processing batch 6/17 (10 chunks)...
Processing batch 7/17 (10 chunks)...
Processing batch 8/17 (10 chunks)...
Processing batch 9/17 (10 chunks)...
Processing batch 10/17 (10 chunks)...
Processing batch 11/17 (10 chunks)...
Process

## 3. Hypothetical Document Embeddings (HyDE)
Before retrieving documents, the engine creates a 'hypothetical answer' mimicking how an official bank policy would respond. This improves search relevance.

In [4]:
test_query = "Is there a formal policy for the calculation of Expected Credit Loss / IFRS 9 approved by Senior Management or the governing body?"

print(f"Original Query: \n{test_query}\n")

print("Generating HyDE Query...")
hyde_query = rag.generate_search_query(test_query)

print("\n=== Generated HyDE Query ===")
print(hyde_query)

Original Query: 
Is there a formal policy for the calculation of Expected Credit Loss / IFRS 9 approved by Senior Management or the governing body?

Generating HyDE Query...

=== Generated HyDE Query ===
La entidad ha establecido una política formal para la cálculo de la Pérdida Esperada de Crédito (ECL) conforme a la norma IFRS 9, la cual ha sido aprobada por la Alta Dirección y el órgano de gobierno. Esta política detalla los métodos utilizados para la estimación de las ECL, incluyendo el enfoque de pérdidas esperadas a 12 meses y el enfoque de pérdidas esperadas de toda la vida, así como la segmentación de la cartera de créditos en grupos homogéneos para una evaluación más precisa. Se implementan procedimientos de validación, como la validación fuera de tiempo (out-of-time validation) y la estabilidad de los modelos (model stability), para asegurar la robustez de las estimaciones. Además, se realizan pruebas de retroceso (backtesting) y análisis de sensibilidad (sensitivity analysis

## 4. Retrieval Process
Finally, we pass the query to the `retrieve` method. Under the hood, this method combines the vector search from both the client index and regulations index based on the HyDE output.

In [5]:
print(f"Retrieving documents for query...\n")
results = rag.retrieve(test_query, k=5)

print(f"Found {len(results)} relevant documents.\n")

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Page: {doc.metadata.get('page', 'Unknown')}")
    print(f"Source Type: {doc.metadata.get('source_type', 'client_doc')}")
    print("Snippet:")
    # Print first 250 characters of the document snippet
    print(f"{doc.page_content[:250]}...\n")


Retrieving documents for query...

DEBUG: Generating HyDE query in Spanish...
Original Query: Is there a formal policy for the calculation of Ex...
HyDE Search Query: La entidad ha establecido una política formal para...
Found 10 relevant documents.

--- Result 1 ---
Source: C:\Users\semeier\Desktop\AI Project\Agentic_AI_for_validation\documents\Política de Previsionamiento_modificada sep 2025.pdf
Page: 2
Source Type: client_doc
Snippet:
Política de Previsionamiento 
 
 
Política de Previsionamiento_modificada sep 2025  Página 3 de 23 
Todos los derechos reservados. Prohibida la reproducción y/o redistribución de su contenido sin el previo consentimiento escrito de BANCO BICA S.A. 
3...

--- Result 2 ---
Source: C:\Users\semeier\Desktop\AI Project\Agentic_AI_for_validation\documents\Política de Previsionamiento_modificada sep 2025.pdf
Page: 1
Source Type: client_doc
Snippet:
Política de Previsionamiento 
 
 
Política de Previsionamiento_modificada sep 2025  Página 2 de 23 
Todos los de

In [6]:
results

[Document(id='a499e572-7620-4b9f-8f5d-af2af2824d56', metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-11-25T13:15:38+01:00', 'title': '', 'author': 'Gabriel Jullier', 'moddate': '2025-11-25T13:15:38+01:00', 'source': 'C:\\Users\\semeier\\Desktop\\AI Project\\Agentic_AI_for_validation\\documents\\Política de Previsionamiento_modificada sep 2025.pdf', 'total_pages': 23, 'page': 2, 'page_label': '3'}, page_content='Política de Previsionamiento \n \n \nPolítica de Previsionamiento_modificada sep 2025  Página 3 de 23 \nTodos los derechos reservados. Prohibida la reproducción y/o redistribución de su contenido sin el previo consentimiento escrito de BANCO BICA S.A. \n3. Aprobar el monto de la pérdida crediticia esperada del segmento considerado como \nde “Tratamiento individual”. \n4. Aprobar las estructuras organizativas para la gestión de los riesgos de incobrabilidad. \n \n3.2 Gerencia de Riesgo y Cumplimiento